# Aula 04 — da pergunta aos dados admissíveis
O notebook inicia com a amostra sintética de famílias da Paraíba (`dados_pof_pb_simulados.csv`). Cada linha representa uma unidade familiar. O exercício é preditivo e não permite interpretação causal. Para usar a RAIS ou dados do projeto, altere a configuração e revise o contrato.

## 1. Configuração e contrato

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEMENTE = 20261001
CAMINHO_DADOS = Path('dados_pof_pb_simulados.csv')
SEPARADOR_CSV = ','
CHAVE = ['id_familia']
ALVO = 'consumo_mensal'
COLUNA_TEMPO = None
COLUNA_GRUPO_PARTICAO = 'municipio'
COLUNAS_NUMERICAS = ['renda_mensal', 'idade_chefe', 'pessoas_familia']
COLUNAS_CATEGORICAS = ['sexo_chefe', 'estado_civil_chefe', 'escolaridade_chefe', 'zona', 'estado']
PREDITOR_BIVARIADO = 'renda_mensal'
VARIAVEL_GRUPO_RESUMO = 'zona'
USAR_LOG1P_NO_ALVO = False
COLUNAS_PROIBIDAS = {}

CONTRATO = {
    'cada_linha_representa': 'uma unidade familiar em um mês de referência',
    'populacao_de_interesse': 'famílias residentes na Paraíba',
    'populacao_registrada': '96 famílias inteiramente simuladas para fins didáticos',
    'instante_da_previsao_t0': 'início do mês de referência',
    'horizonte_do_alvo': 'consumo total observado ao fim do mês',
    'uso_simulado_pelo_teste': 'prever consumo de famílias não usadas no ajuste',
}
pd.Series(CONTRATO, name='definicao')

## 2. Leitura e validação da unidade
A leitura ocorre antes de qualquer resumo. As asserções testam parte do contrato, mas não substituem o dicionário.

In [ ]:
DADOS_DISPONIVEIS = CAMINHO_DADOS.exists()
if DADOS_DISPONIVEIS:
    dados = pd.read_parquet(CAMINHO_DADOS) if CAMINHO_DADOS.suffix.lower() == '.parquet' else pd.read_csv(CAMINHO_DADOS, sep=SEPARADOR_CSV, low_memory=False)
    assert CHAVE, 'Defina CHAVE conforme a unidade de análise.'
    assert ALVO, 'Defina ALVO conforme a pergunta preditiva.'
    preditoras = COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS
    esperadas = set(CHAVE + [ALVO] + preditoras)
    for opcional in [COLUNA_TEMPO, COLUNA_GRUPO_PARTICAO, PREDITOR_BIVARIADO, VARIAVEL_GRUPO_RESUMO]:
        if opcional: esperadas.add(opcional)
    ausentes = esperadas - set(dados.columns)
    assert not ausentes, f'Colunas não encontradas: {sorted(ausentes)}'
    assert dados[CHAVE].notna().all().all(), 'Há valor ausente na chave.'
    assert not dados.duplicated(CHAVE).any(), 'Há chave duplicada; revise unidade e granularidade.'
    print(f'{len(dados):,} linhas; {dados.shape[1]} colunas; chave validada.')
else:
    dados = None
    print('Base não localizada:', CAMINHO_DADOS)
    print('Preencha o contrato e marque os resultados empíricos como não verificados.')

## 3. Qualidade, cobertura e distribuição do alvo
Mantenha contagem e denominador. Na amostra POF simulada, compare renda e consumo em escala original e examine assimetria, valores extremos e ausências em escolaridade.

In [ ]:
if DADOS_DISPONIVEIS:
    import matplotlib.pyplot as plt
    perfil = pd.DataFrame({
        'tipo': dados.dtypes.astype(str), 'n_observado': dados.notna().sum(),
        'n_faltante': dados.isna().sum(), 'taxa_faltante': dados.isna().mean(),
        'n_unicos': dados.nunique(dropna=False),
    }).sort_values('taxa_faltante', ascending=False)
    display(perfil)
    recorte = COLUNA_TEMPO or VARIAVEL_GRUPO_RESUMO
    if recorte:
        cobertura = dados.groupby(recorte, dropna=False)[ALVO].agg(n='size', n_alvo='count')
        cobertura['taxa_alvo_ausente'] = 1 - cobertura['n_alvo'] / cobertura['n']
        display(cobertura)
    alvo_bruto = pd.to_numeric(dados[ALVO], errors='coerce')
    display(alvo_bruto.describe(percentiles=[.01,.25,.5,.75,.95,.99]).to_frame(ALVO))
    fig, eixos = plt.subplots(1, 2, figsize=(11, 3.6))
    alvo_bruto.plot.hist(bins=40, ax=eixos[0], edgecolor='white', title=f'Distribuição de {ALVO}')
    np.log1p(alvo_bruto.clip(lower=0)).plot.hist(bins=40, ax=eixos[1], edgecolor='white', title=f'log1p({ALVO})')
    for ax in eixos: ax.set_ylabel('Número de unidades')
    plt.tight_layout()
else:
    print('Perfil e distribuição não calculados: falta a base.')

## 4. Relações com o alvo
O gráfico antecede a reta. Examine forma, suporte, dispersão vertical e diferenças entre grupos. Correlação não demonstra linearidade nem causalidade.

In [ ]:
if DADOS_DISPONIVEIS and PREDITOR_BIVARIADO:
    y_exp = np.log1p(pd.to_numeric(dados[ALVO], errors='coerce').clip(lower=0)) if USAR_LOG1P_NO_ALVO else pd.to_numeric(dados[ALVO], errors='coerce')
    x_exp = pd.to_numeric(dados[PREDITOR_BIVARIADO], errors='coerce')
    pares = pd.DataFrame({'x': x_exp, 'y': y_exp}).dropna()
    amostra = pares.sample(min(5000, len(pares)), random_state=SEMENTE)
    ax = amostra.plot.scatter(x='x', y='y', alpha=.18, s=12, figsize=(7,4))
    ax.set(xlabel=PREDITOR_BIVARIADO, ylabel=('log1p('+ALVO+')') if USAR_LOG1P_NO_ALVO else ALVO)
    print('Correlação de Pearson:', pares.corr().loc['x','y'].round(3))
else:
    print('Dispersão não gerada: defina PREDITOR_BIVARIADO e disponibilize a base.')

if DADOS_DISPONIVEIS and VARIAVEL_GRUPO_RESUMO:
    resumo_grupo = dados.groupby(VARIAVEL_GRUPO_RESUMO, dropna=False)[ALVO].agg(
        n='size', media='mean', mediana='median', desvio='std',
        q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75))
    display(resumo_grupo.sort_values('n', ascending=False))

**Registro da equipe:** qual pergunta cada saída respondeu, que risco revelou e qual decisão decorre dela?

## 5. Relógio da decisão e leakage
Registre geração, entrada no sistema, disponibilidade em $t_0$ e decisão permitir/proibir para ao menos cinco variáveis.

In [ ]:
tabela_leakage = pd.DataFrame([
    {'variavel': c, 'permitida_em_t0': False, 'justificativa': j}
    for c, j in COLUNAS_PROIBIDAS.items()])
display(tabela_leakage)
if DADOS_DISPONIVEIS:
    encontradas = set(COLUNAS_PROIBIDAS) & set(dados.columns)
    assert not (encontradas & set(COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS)), 'Uma coluna proibida foi incluída como preditora.'
    print('Colunas proibidas presentes na base bruta:', sorted(encontradas))

## 6. Partição que representa o uso
Datas completas e entidades inteiras permanecem em apenas um lado da divisão.

In [ ]:
if DADOS_DISPONIVEIS:
    from sklearn.model_selection import GroupShuffleSplit, train_test_split
    preditoras = COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS
    assert preditoras, 'Defina ao menos uma preditora.'
    X = dados[preditoras].copy()
    y0 = pd.to_numeric(dados[ALVO], errors='coerce')
    y = np.log1p(y0.clip(lower=0)) if USAR_LOG1P_NO_ALVO else y0
    validos = y.notna(); X, y, base = X.loc[validos], y.loc[validos], dados.loc[validos]
    if COLUNA_TEMPO:
        datas = pd.to_datetime(base[COLUNA_TEMPO], errors='raise')
        unicas = np.sort(datas.dropna().unique()); assert len(unicas) >= 2
        corte = unicas[max(1, int(.75*len(unicas)))-1]
        idx_tr, idx_te = datas.index[datas <= corte], datas.index[datas > corte]
        assert datas.loc[idx_tr].max() < datas.loc[idx_te].min()
        estrategia = f'temporal; treino até {pd.Timestamp(corte).date()}'
    elif COLUNA_GRUPO_PARTICAO:
        div = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEMENTE)
        pos_tr, pos_te = next(div.split(X, y, groups=base[COLUNA_GRUPO_PARTICAO]))
        idx_tr, idx_te = X.index[pos_tr], X.index[pos_te]
        assert set(base.loc[idx_tr,COLUNA_GRUPO_PARTICAO]).isdisjoint(set(base.loc[idx_te,COLUNA_GRUPO_PARTICAO]))
        estrategia = f'por grupo: {COLUNA_GRUPO_PARTICAO}'
    else:
        idx_tr, idx_te = train_test_split(X.index, test_size=.25, random_state=SEMENTE)
        estrategia = 'aleatória; justificar independência das linhas'
    X_treino, X_teste = X.loc[idx_tr], X.loc[idx_te]
    y_treino, y_teste = y.loc[idx_tr], y.loc[idx_te]
    print(estrategia, '| treino:', X_treino.shape, '| teste:', X_teste.shape)
else:
    X_treino = X_teste = y_treino = y_teste = None

## 7. Baseline fora da amostra
O baseline prevê a média do treino. Ele estabelece o erro que a regressão deverá superar.

In [ ]:
if DADOS_DISPONIVEIS:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    pred_base = np.repeat(y_treino.mean(), len(y_teste))
    mae_base = mean_absolute_error(y_teste, pred_base)
    rmse_base = mean_squared_error(y_teste, pred_base) ** .5
    print(f'Baseline da média | MAE={mae_base:.4f} | RMSE={rmse_base:.4f}')
else:
    print('Baseline não calculado: falta a base.')

## 8. Pré-processamento ajustado somente no treino
Medianas, modas, categorias e escalas são aprendidas no treino. O teste recebe apenas `transform`.

In [ ]:
if DADOS_DISPONIVEIS:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    transformadores = []
    if COLUNAS_NUMERICAS:
        transformadores.append(('num', Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]), COLUNAS_NUMERICAS))
    if COLUNAS_CATEGORICAS:
        transformadores.append(('cat', Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),('codificar',OneHotEncoder(handle_unknown='ignore'))]), COLUNAS_CATEGORICAS))
    preparo = ColumnTransformer(transformadores)
    Xtr = preparo.fit_transform(X_treino); Xte = preparo.transform(X_teste)
    assert Xtr.shape[1] == Xte.shape[1]
    print('Matriz de treino:', Xtr.shape, '| matriz de teste:', Xte.shape)
else:
    print('Pré-processamento não ajustado: falta a base.')

## Saídas da equipe
1. Contrato e unidade.  
2. Distribuição do alvo nas duas escalas.  
3. Relação com o alvo ligada a uma decisão.  
4. Qualidade com número, taxa e recorte.  
5. Tabela de leakage.  
6. Partição coerente com o uso.  
7. MAE e RMSE do baseline no teste.  
8. Evidência de `fit` somente no treino.